# Diagnostic · What would the tester miss?

Give yourself 35 minutes. Work from the tiny inputs and predict the answers on paper before coding. These five tasks isolate mistakes that can survive a batch-shape check or a plausible accuracy. Use the outcome to choose which exercise to revisit. No training or data files are needed.

In [1]:
import copy
import numpy as np
import torch
from torch import nn

## 1 · Recover a scrambled measurement · 6 min

The first export has one row, and the second export has two. `stacked` should have three rows in the first export’s column order. Complete `combine` using array operations so it also works when the second export is permuted differently.

In [18]:
def combine(a, ha, b, hb):
    order = (ha[:, None] == hb).argmax(axis=1)
    return np.concatenate((a, b[:, order]))

ha, hb = np.array(["mass", "gloss", "score"]), np.array(["score", "mass", "gloss"])
a = np.array([[8, 0.3, 4]])
b = np.array([[7, 10, 0.6], [2, 9, 0.4]])
stacked = combine(a, ha, b, hb)

In [19]:
# Check your work
expected = [[8, .3, 4], [10, .6, 7], [9, .4, 2]]
assert np.allclose(stacked, expected)
assert np.allclose(combine(a, ha, b[:, ::-1], hb[::-1]), expected)
print("Check passed.")

Check passed.


In [20]:
scores = np.array([[3, 5, 7], [3, 6, 7], [2, 8, 9], [4, 7, 7], [5, 5, 8], [5, 5, 7]])

## 2 · Switch the label rule · 7 min

Apply the two **released** rules to the same score matrix. For A, premium means average > 5 and minimum ≥ 3. For B, bottom means any score ≤ 2, top means minimum ≥ 5 and average ≥ 6, middle covers the rest. Create `ya` as a float32 tensor of shape (6, 1) and `yb` as int64 of shape (6,). Explain what each boundary row tests.

In [43]:
# Create ya and yb.
premium = np.mean(scores, axis = 1) > 5
print(premium)
print()

minimum = np.mean(scores, axis = 1) >= 3
print(minimum)
print()

ya = torch.tensor(premium & minimum, dtype=torch.float32)
print(ya.tolist())


True

[ True  True  True  True  True  True]

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


In [41]:
# Check your work
assert ya.dtype == torch.float32 and ya.tolist() == [[0], [1], [0], [1], [1], [1]]
assert yb.dtype == torch.int64 and yb.tolist() == [1, 1, 0, 1, 2, 1]
print("Check passed.")

AssertionError: 

In [31]:
batch_sizes = np.array([32, 32, 1])
batch_mean_losses = np.array([0.2, 0.3, 1.5])
probe = nn.Linear(1, 1, bias=False)
with torch.no_grad():
    probe.weight.fill_(2.0)

## 3 · Rescue the validation checkpoint · 6 min

The validation loss was best when the weight was 2. Save a snapshot as `best`, simulate one more optimizer update by setting the weight to 9, then restore. Print the weight before and after restoration. Also compute `test_loss` across all 65 samples from the supplied batch summaries.

In [ ]:
# Save, simulate an update, restore and compute the weighted loss.

In [ ]:
# Check your work
assert probe.weight.item() == best['weight'].item() == 2.0
assert np.isclose(test_loss, 17.5/65)
print("Check passed.")

In [ ]:
# Each tuple is one test batch, in the order yielded.
batches = [
    (np.array([42, 17]), np.array([0.03, 0.76]), np.array([0, 1])),
    (np.array([90, 26]), np.array([0.51, 0.46]), np.array([0, 0])),
    (np.array([81]), np.array([0.60]), np.array([1])),
]

## 4 · Find uncertainty across batches · 6 min

Find the two least confident binary decisions across the entire loader. Return `ids, probabilities, labels` for those two in increasing order of distance to the boundary. Keep records aligned. Explain why the most confidently wrong example would answer a different question.

In [ ]:
# Gather records, rank and index.

In [ ]:
# Check your work
assert np.array_equal(ids, [90, 26])
assert np.allclose(probabilities, [.51, .46]) and np.array_equal(labels, [0, 0])
print("Check passed.")

In [ ]:
y = np.array([0, 0, 0, 0, 1, 2])
predicted = np.array([0, 0, 1, 0, 0, 1])

## 5 · Compute what the prompt defines · 10 min

Compute a 3 × 3 confusion matrix `cm` with true rows, the ordinary accuracy `acc`, the mean per-class recall `mean_recall` and the exam’s mean one-vs-rest accuracy `macro`. Use shared operations across classes. Work out why one wrong prediction counts as two failed binary decisions out of three.

In [ ]:
# Compute and print all four outputs.

In [ ]:
# Check your work
assert np.array_equal(cm, [[3, 1, 0], [1, 0, 0], [0, 1, 0]])
assert np.allclose([acc, mean_recall, macro], [.5, .25, 2/3])
print("Check passed.")

## Use your result

If 1 or 2 cost you the most time, do the data sections of both new mocks first. If 3 did, revisit exercises 5.8 and 9.2–9.3 before training another model. If 4 or 5 did, work directly on the final sections of the mocks using the small examples here. If you finished all five unaided, move straight to a timed released paper.